In [0]:
#====================================================================================
# VIDA CRM Lakehouse
# Gold Layer - Star Schema
# Author: Emmanuel Quesada Gómez
#====================================================================================

from pyspark.sql.functions import col, sum, avg, count, round

SILVER_PATH = "workspace.default"
GOLD_PATH = "workspace.default"

print("Libraries loaded successfully")

Libraries loaded successfully


In [0]:
#====================================================================================
# Gold - dim_volunteer
#====================================================================================

df_silver_volunteer = spark.table(f"{SILVER_PATH}.silver_volunteer")

df_gold_volunteer = df_silver_volunteer.select(
    "volunteer_id",
    "first_name",
    "last_name",
    "email",
    "gender",
    "nationality",
    "birth_date",
    "start_date",
    "status",
    "education_level"
)

df_gold_volunteer.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.gold_volunteer")
print(f"gold_volunteer: {df_gold_volunteer.count()} records")

gold_volunteer: 8000 records


In [0]:
#====================================================================================
# Gold - dim_geography
#====================================================================================

df_gold_geography = spark.table(f"{SILVER_PATH}.silver_geography").select(
    "geography_id",
    "country",
    "region",
    "capital_city",
    "city",
    "rural_urban"
)

df_gold_geography.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.gold_geography")
print(f"gold_geography: {df_gold_geography.count()} records")

gold_geography: 250 records


In [0]:
#====================================================================================
# Gold - dim_organization
#====================================================================================

df_gold_organization = spark.table(f"{SILVER_PATH}.silver_organization").select(
    "organization_id",
    "organization_name",
    "org_type",
    "geography_id",
    "founded_year",
    "active",
)

df_gold_organization.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.gold_organization")
print(f"gold_organization: {df_gold_organization.count()} records")

gold_organization: 800 records


In [0]:
#====================================================================================
# Gold - dim_project
#====================================================================================

df_gold_project = spark.table(f"{SILVER_PATH}.silver_project").select(
    "project_id",
    "project_name",
    "project_type",
    "geography_id",
    "organization_id",
    "start_date",
    "end_date",
    "budget_usd",
    "status"
)

df_gold_project.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.gold_project")
print(f"gold_project: {df_gold_project.count()} records")

gold_project: 400 records


In [0]:
#====================================================================================
# Gold - dim_date
#====================================================================================

df_gold_date = spark.table(f"{SILVER_PATH}.silver_date").select(
    "date_id",
    "full_date",
    "year",
    "quarter",
    "month",
    "month_name",
    "week",
    "day_of_week",
    "is_weekend"
)

df_gold_date.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.gold_date")
print(f"gold_date: {df_gold_date.count()} records")

gold_date: 3653 records


In [0]:
#====================================================================================
# Gold - fact_volunteer_activity
#====================================================================================

df_silver_fact = spark.table(f"{SILVER_PATH}.silver_fact_activity")
df_silver_volunteer = spark.table(f"{SILVER_PATH}.silver_volunteer")
df_silver_project = spark.table(f"{SILVER_PATH}.silver_project")
df_silver_geography = spark.table(f"{SILVER_PATH}.silver_geography")

df_gold_fact = df_silver_fact \
    .join(df_silver_volunteer.select("volunteer_id", "status", "education_level"), "volunteer_id", "left") \
    .join(df_silver_project.select("project_id", "project_type", "budget_usd"), "project_id", "left") \
    .join(df_silver_geography.select("geography_id", "country", "region"), "geography_id", "left") \
    .select(
        "activity_id",
        "volunteer_id",
        "project_id",
        "organization_id",
        "geography_id",
        "date_id",
        "activity_type",
        "hours_logged",
        "beneficiaries_reached",
        "outcome",
        "status",
        "education_level",
        "project_type",
        "budget_usd",
        "country",
        "region"
    ) 

df_gold_fact.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_PATH}.gold_fact_activity")
print(f"gold_fact_activity: {df_gold_fact.count()} records")

gold_fact_activity: 800000 records
